[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Pandas, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/pandas-deep-dive.html)

# Reading Data


## What you will be able to do

Read CSV files into tables with `read_csv`, and choose the arguments that make every column come out
with the right name and type, whatever separator, decimal mark, encoding or missing-value marker the
file uses.

## The idea

### The problem

The readings in this guide are built by a formula in Setup. Real readings arrive as files, and every
program that writes a file makes its own choices. A spreadsheet set to Norwegian writes semicolons
between the values and commas inside the decimals, so -3.5 becomes `-3,5`. A logger writes `--` where
it had no reading. An older program saves Tromsø with an encoding other than UTF-8, and Excel's "CSV
UTF-8" puts three invisible bytes in front of the first column's name. A station number such as `0101`
looks like a number and is not one.

`read_csv` reads all of these, and for most of them it reads them without an error. A file split at the
wrong character comes back as one column. A column of decimals with commas in them comes back as text,
and so does a column of numbers with one `--` among them. The ids lose their leading zeros. Each table
prints, and each one gives wrong answers later, so the work is in knowing which argument each file needs
and in checking the result's columns and types before going on.

### What read_csv is

> **`read_csv`** reads a delimited text file into a DataFrame. It splits each line at a **separator**,
> takes the first line as the column names, and **infers** a type for each column from its values,
> unless its arguments say otherwise.

### Why it works that way

A CSV file holds only text. Every type in the result is pandas' guess from that text, made column by
column. A column whose every value reads as a whole number becomes `int64`, which is why `0101` becomes
101. A column whose values all read as decimals becomes `float64`. Anything else, a single `-3,5` or `--`
included, leaves the whole column as text, `object`. Dates stay text too unless `parse_dates` names them.

A few strings count as missing without being asked, among them an empty field, `NA`, `NaN`, `n/a` and
`null`. A marker outside that list, such as `--`, is text until `na_values` adds it. The separator is a
comma unless `sep` says otherwise, the decimal mark is a point unless `decimal` says otherwise, and the
bytes are decoded as UTF-8 unless `encoding` says otherwise.

So the arguments that matter are the ones that replace a guess: `sep`, `decimal`, `encoding`,
`na_values`, `dtype` and `parse_dates` for what the file means, and `usecols`, `nrows` and `index_col`
for what you want from it.

### Where this shows up

`read_csv` reads data that comes from outside the program, such as exports from spreadsheets and databases,
files from instruments and downloads from public data portals. The **CSV** notebook in the **Files, Paths and
Formats** guide read the same kind of file a row at a time with the `csv` module, and its **Encodings**
notebook explains what UTF-8 and Latin-1 are. `read_csv` does the same reading in one call and returns a
table.

### What this notebook covers

- reading a file, and checking its types with `dtypes`
- dates with `parse_dates`, an index with `index_col`, and part of a file with `usecols` and `nrows`
- a column kept as text with `dtype`, so that ids keep their leading zeros
- semicolons and decimal commas with `sep` and `decimal`
- a missing-value marker with `na_values`, and an encoding with `encoding`
- the byte order mark Excel writes, and what reading it with the wrong encoding does

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read it,
and read the output underneath it. Everything from Setup onward is where you start running things.

```python
import io
import pandas as pd

text = "station;celsius\nOslo;-3,5\nBergen;-3,6\n"
print(pd.read_csv(io.StringIO(text), sep=";", decimal=","))
```

```
  station  celsius
0    Oslo     -3.5
1  Bergen     -3.6
```

`io.StringIO` lets a string stand in for a file. The text uses semicolons between values and commas in
the decimals, and two arguments said so, so the readings came out as numbers.

## Setup

One library, the helpers that build the readings, and the files this notebook reads.

- `pandas`, imported as `pd`, is the library, and the cell prints its version
- `math`, `datetime` and `timedelta` build the readings
- `Path` names the files in the `scratch` folder
- `shutil` removes the `scratch` folder at the end of Common errors
- `warnings` catches the warning `read_csv` gives for a line it skips, in Common errors, so the cell can
  print it

`readings` and `stations` are the guide's two tables, the 35,040 hourly readings at four stations and
each station's latitude. The cell writes them to files the way other programs would, all into `scratch`:

- `readings.csv`, the whole year, written by pandas with commas
- `stations.csv`, the stations with a made-up four-digit id, `0101` to `0105`
- `stations_excel.csv`, the same file with the byte order mark that Excel's "CSV UTF-8" adds
- `stations_latin1.csv`, the same stations with Tromsø spelled with its `ø`, saved in Latin-1
- `oslo_export.csv`, Oslo's first six hours with semicolons and decimal commas
- `svalbard_export.csv`, five of Svalbard's hours around its silent 2 March, with `--` where it sent
  nothing
- `hand_edited.csv`, three readings, one of which somebody edited by hand

In [1]:
import math
import shutil
import warnings
from datetime import datetime, timedelta
from pathlib import Path

import pandas as pd

STATIONS = {"Bergen": 8.0, "Oslo": 6.5, "Svalbard": -4.5, "Tromso": 3.5}   # each station's mean for the year
LATITUDES = {"Bergen": 60.39, "Oslo": 59.91, "Svalbard": 78.22, "Tromso": 69.65, "Kirkenes": 69.73}


def year_of_readings():
    """Every hour of 2025 at the four stations, as the sqlite3 guide makes them, with Svalbard silent on 2 March."""
    for n in range(365 * 24):
        hour = datetime(2025, 1, 1) + timedelta(hours=n)
        season = -math.cos(2 * math.pi * (n - 400) / (365 * 24))
        day = -math.cos(2 * math.pi * (hour.hour - 3) / 24)
        for i, (station, mean) in enumerate(STATIONS.items()):
            if station == "Svalbard" and hour.strftime("%Y-%m-%d") == "2025-03-02":
                celsius = None
            else:
                wobble = ((n * 37 + i * 101) % 17 - 8) / 10
                celsius = round(mean + 9 * season + 3 * day + wobble, 1) + 0.0
            yield station, hour.strftime("%Y-%m-%dT%H:%M"), celsius


readings = pd.DataFrame(list(year_of_readings()), columns=["station", "time", "celsius"])
readings["time"] = pd.to_datetime(readings["time"], format="%Y-%m-%dT%H:%M")
stations = pd.DataFrame({"station": list(LATITUDES), "latitude": list(LATITUDES.values())})

SCRATCH = Path("scratch")
SCRATCH.mkdir(exist_ok=True)
numbered = stations.assign(id=["0101", "0102", "0103", "0104", "0105"])[["id", "station", "latitude"]]
oslo_hours = readings[readings["station"] == "Oslo"].head(6)
svalbard_hours = readings[(readings["station"] == "Svalbard")
                          & readings["time"].between("2025-03-01 22:00", "2025-03-02 02:00")]

readings.to_csv(SCRATCH / "readings.csv", index=False)
numbered.to_csv(SCRATCH / "stations.csv", index=False)
numbered.to_csv(SCRATCH / "stations_excel.csv", index=False, encoding="utf-8-sig")
numbered.replace({"Tromso": "Tromsø"}).to_csv(SCRATCH / "stations_latin1.csv", index=False, encoding="latin-1")
oslo_hours.to_csv(SCRATCH / "oslo_export.csv", sep=";", decimal=",", index=False, date_format="%Y-%m-%d %H:%M")
svalbard_hours.to_csv(SCRATCH / "svalbard_export.csv", index=False, na_rep="--", date_format="%Y-%m-%d %H:%M")
(SCRATCH / "hand_edited.csv").write_text("station,time,celsius\n"
                                         "Bergen,2025-01-01 00:00,-3.6\n"
                                         "Oslo,2025-01-01 00:00,-3.5,checked\n"
                                         "Svalbard,2025-01-01 00:00,-14.6\n")

print("pandas", pd.__version__, "|", sorted(path.name for path in SCRATCH.iterdir()))

pandas 2.2.3 | ['hand_edited.csv', 'oslo_export.csv', 'readings.csv', 'stations.csv', 'stations_excel.csv', 'stations_latin1.csv', 'svalbard_export.csv']


## Worked examples

### Reading a file

`read_csv` takes a path and returns a table. Here are the first lines of `readings.csv` as text, and the
table they become:

In [2]:
print((SCRATCH / "readings.csv").read_text().splitlines()[:3])

from_file = pd.read_csv(SCRATCH / "readings.csv")
print(from_file.shape)
print(from_file.dtypes)

['station,time,celsius', 'Bergen,2025-01-01 00:00:00,-3.6', 'Oslo,2025-01-01 00:00:00,-3.5']
(35040, 3)
station     object
time        object
celsius    float64
dtype: object


All 35,040 rows came back, and `celsius` is `float64`, because every value in it reads as a decimal or is
empty, and an empty field is missing. `time` is `object`, text, because nothing told `read_csv` that the
column holds dates.

### Dates with parse_dates

In [3]:
from_file = pd.read_csv(SCRATCH / "readings.csv", parse_dates=["time"])

print(from_file.dtypes)
print(from_file["celsius"].isna().sum(), "readings missing")

station            object
time       datetime64[ns]
celsius           float64
dtype: object
24 readings missing


`parse_dates` takes a list of columns to read as dates and times. The 24 missing readings are Svalbard's
2 March, written as empty fields and read back as `NaN`.

### Part of a file with usecols and nrows

`usecols` reads only the columns named, and `nrows` stops after that many rows:

In [4]:
pd.read_csv(SCRATCH / "readings.csv", usecols=["station", "celsius"], nrows=8)

,station,celsius
0,Bergen,-3.6
1,Oslo,-3.5
2,Svalbard,-14.6
3,Tromso,-6.7
4,Bergen,-3.7
5,Oslo,-5.3
6,Svalbard,-16.4
7,Tromso,-8.5


Both leave out what a question does not need, and `nrows` is a cheap way to look at the
start of a file before reading all of it.

### An index from a column with index_col

In [5]:
by_time = pd.read_csv(SCRATCH / "readings.csv", parse_dates=["time"], index_col="time")

print(by_time.index[:3])
print(by_time.columns.tolist())

DatetimeIndex(['2025-01-01', '2025-01-01', '2025-01-01'], dtype='datetime64[ns]', name='time', freq=None)
['station', 'celsius']


The `time` column became the index, an index of times, and left the columns. **The Index** notebook
covers what an index is for.

### Ids kept as text with dtype

`stations.csv` numbers its stations `0101` to `0105`:

In [6]:
print((SCRATCH / "stations.csv").read_text())

pd.read_csv(SCRATCH / "stations.csv", dtype={"id": str})

id,station,latitude
0101,Bergen,60.39
0102,Oslo,59.91
0103,Svalbard,78.22
0104,Tromso,69.65
0105,Kirkenes,69.73



,id,station,latitude
0,0101,Bergen,60.39
1,0102,Oslo,59.91
2,0103,Svalbard,78.22
3,0104,Tromso,69.65
4,0105,Kirkenes,69.73


`dtype` takes a dictionary of column names and types. `str` keeps every id exactly as the file wrote it,
leading zero included. Without it, the ids read as numbers, which Common errors shows.

### Semicolons and decimal commas with sep and decimal

In [7]:
print((SCRATCH / "oslo_export.csv").read_text())

oslo = pd.read_csv(SCRATCH / "oslo_export.csv", sep=";", decimal=",")
print(oslo.dtypes)
print("mean:", round(oslo["celsius"].mean(), 2))

station;time;celsius
Oslo;2025-01-01 00:00;-3,5
Oslo;2025-01-01 01:00;-5,3
Oslo;2025-01-01 02:00;-5,3
Oslo;2025-01-01 03:00;-5,1
Oslo;2025-01-01 04:00;-4,7
Oslo;2025-01-01 05:00;-4,1

station     object
time        object
celsius    float64
dtype: object
mean: -4.67


`sep=";"` splits each line at the semicolons, and `decimal=","` reads `-3,5` as -3.5, so `celsius` is
`float64` and has a mean.

### A missing-value marker with na_values

In [8]:
print((SCRATCH / "svalbard_export.csv").read_text())

svalbard = pd.read_csv(SCRATCH / "svalbard_export.csv", na_values="--")
print(svalbard)
print(svalbard.dtypes)

station,time,celsius
Svalbard,2025-03-01 22:00,-11.6
Svalbard,2025-03-01 23:00,-12.0
Svalbard,2025-03-02 00:00,--
Svalbard,2025-03-02 01:00,--
Svalbard,2025-03-02 02:00,--

    station              time  celsius
0  Svalbard  2025-03-01 22:00    -11.6
1  Svalbard  2025-03-01 23:00    -12.0
2  Svalbard  2025-03-02 00:00      NaN
3  Svalbard  2025-03-02 01:00      NaN
4  Svalbard  2025-03-02 02:00      NaN
station     object
time        object
celsius    float64
dtype: object


`na_values` adds `--` to the strings that count as missing, so the three silent hours are `NaN` and the
column is `float64`. It takes one string, a list, or a dictionary of column names and markers when a
marker means missing in only one column.

### An encoding with encoding

Printed as bytes, the Latin-1 file shows Tromsø's `ø` as the single byte `\xf8`. UTF-8 writes the same
character as two bytes, which the first line prints:

In [9]:
print("ø".encode("utf-8"))
print((SCRATCH / "stations_latin1.csv").read_bytes())

pd.read_csv(SCRATCH / "stations_latin1.csv", encoding="latin-1", dtype={"id": str})

b'\xc3\xb8'
b'id,station,latitude\n0101,Bergen,60.39\n0102,Oslo,59.91\n0103,Svalbard,78.22\n0104,Troms\xf8,69.65\n0105,Kirkenes,69.73\n'


,id,station,latitude
0,0101,Bergen,60.39
1,0102,Oslo,59.91
2,0103,Svalbard,78.22
3,0104,Tromsø,69.65
4,0105,Kirkenes,69.73


`encoding="latin-1"` decodes each byte as one character, and `ø` comes back. The **Encodings** notebook in
the **Files, Paths and Formats** guide explains the difference between the two encodings.

### The byte order mark Excel writes

Excel's "CSV UTF-8" starts the file with three bytes, `\xef\xbb\xbf`, called a **byte order mark**. Read
as UTF-8, which is `read_csv`'s default, they disappear:

In [10]:
print((SCRATCH / "stations_excel.csv").read_bytes()[:12])

pd.read_csv(SCRATCH / "stations_excel.csv", dtype={"id": str}).columns.tolist()

b'\xef\xbb\xbfid,statio'


['id', 'station', 'latitude']

The first column is named `id`, as it should be. Common errors shows the same file read with the wrong
encoding.

## Your turn

Five tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than
getting there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pandas-deep-dive/04-reading-data-solutions.ipynb).

**1.** Read `scratch/readings.csv` with `time` read as dates and times, and print the table's dtypes.

In [11]:
# your code here

**2.** Read only the `station` and `celsius` columns of the first 12 rows of `scratch/readings.csv`, and print them.

In [12]:
# your code here

**3.** Read `scratch/oslo_export.csv` so that `celsius` comes out as numbers, and print Oslo's lowest reading in it.

In [13]:
# your code here

**4.** Read `scratch/stations.csv` with the ids kept as text and used as the index, and print the station whose id is `0103`.

In [14]:
# your code here

**5.** Read `scratch/svalbard_export.csv` so that `--` counts as missing, and print how many of its readings are missing.

In [15]:
# your code here

## Common errors

### ParserError: Error tokenizing data. C error: Expected 3 fields in line 3, saw 4

In [16]:
pd.read_csv(SCRATCH / "hand_edited.csv")

ParserError: Error tokenizing data. C error: Expected 3 fields in line 3, saw 4


The header has three names, so `read_csv` expects three fields on every line, and line 3 of the file has
four. Read the line itself to see what happened:

In [17]:
print((SCRATCH / "hand_edited.csv").read_text().splitlines()[2])

Oslo,2025-01-01 00:00,-3.5,checked


Somebody added `checked` to Oslo's reading. Fixing the file is the right answer when you can. When you
cannot, `on_bad_lines="warn"` skips such lines and warns about each one, so you know what was dropped:

In [18]:
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    edited = pd.read_csv(SCRATCH / "hand_edited.csv", on_bad_lines="warn")

print(edited)
for warning in caught:
    print(f"{warning.category.__name__}: {str(warning.message).strip()}")

    station              time  celsius
0    Bergen  2025-01-01 00:00     -3.6
1  Svalbard  2025-01-01 00:00    -14.6


Oslo's reading is gone from the table, and the warning names its line.

### UnicodeDecodeError: 'utf-8' codec can't decode byte 0xf8 in position 84: invalid start byte

In [19]:
pd.read_csv(SCRATCH / "stations_latin1.csv")

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xf8 in position 84: invalid start byte

`read_csv` decodes as UTF-8 unless told otherwise, and byte `0xf8`, Latin-1's `ø` in Tromsø, is not valid
UTF-8 on its own. Position 84 counts bytes from the start of the file. Pass the file's own encoding,
`encoding="latin-1"`, as the worked example did.

### No error, and a column named ï»¿id

In [20]:
excel = pd.read_csv(SCRATCH / "stations_excel.csv", encoding="latin-1")

print(excel.columns.tolist())

['ï»¿id', 'station', 'latitude']


Read as Latin-1, the three bytes of the byte order mark became three characters, `ï»¿`, at the start of
the first column's name. Nothing raised while reading, and the first use of the column by its name fails:

In [21]:
excel["id"]

KeyError: 'id'

The column is called `ï»¿id`, so `id` is not there. A file from Excel's "CSV UTF-8" is UTF-8, so read it
with the default encoding, or with `encoding="utf-8-sig"`, which removes the mark explicitly.

### No error, and the whole file in one column

In [22]:
squashed = pd.read_csv(SCRATCH / "oslo_export.csv")

print(squashed.shape)
print(squashed.columns.tolist())
print(squashed.head(2))

(6, 1)
['station;time;celsius']
                          station;time;celsius
Oslo;2025-01-01 00:00;-3                     5
Oslo;2025-01-01 01:00;-5                     3


Without `sep`, `read_csv` split every line at its commas. The header line has none, so the table has one
column, named for the whole header. Each reading has one comma, its decimal comma, so each row split in
two, and with one field more in the rows than in the header, `read_csv` used the first field as the index.
A single column whose name holds semicolons means the file needs `sep=";"`.

### No error, and the readings read as text

In [23]:
oslo_as_text = pd.read_csv(SCRATCH / "oslo_export.csv", sep=";")

print(oslo_as_text.head(2))
print(oslo_as_text.dtypes)

  station              time celsius
0    Oslo  2025-01-01 00:00    -3,5
1    Oslo  2025-01-01 01:00    -5,3
station    object
time       object
celsius    object
dtype: object


`sep` was right, so the columns came apart, and `decimal` was left out, so `-3,5` is not a number to
`read_csv` and `celsius` stayed text, `object`. The table looks right except for the commas. Asking for a
mean shows the problem:

In [24]:
oslo_as_text["celsius"].mean()

TypeError: Could not convert string '-3,5-5,3-5,3-5,1-4,7-4,1' to numeric

The text in the message is the six values joined end to end. To average a column, pandas first adds its
values, and adding strings joins them, so the sum was one long string, which pandas cannot convert to a
number.
Reading the file with `decimal=","` gives a column of numbers, and its mean, as the worked example did.

### No error, and the ids lost their leading zeros

In [25]:
numbers = pd.read_csv(SCRATCH / "stations.csv")

print(numbers["id"].tolist())
print(numbers.dtypes["id"])

[101, 102, 103, 104, 105]
int64


Every id reads as a whole number, so `read_csv` made the column `int64`, and `0101` became 101. As a
number, 101 keeps no record of the zero the file wrote in front of it. An id is a name, so read it with
`dtype={"id": str}`.

### ValueError: Usecols do not match columns, columns expected but not found: ['Celsius']

In [26]:
pd.read_csv(SCRATCH / "readings.csv", usecols=["station", "Celsius"])

ValueError: Usecols do not match columns, columns expected but not found: ['Celsius']

`usecols` names must match the file's header exactly, case included, and the message lists the names it
could not find, here `Celsius` for the column the file calls `celsius`.

The notebook is finished with its files, so this cell removes the `scratch` folder:

In [27]:
shutil.rmtree(SCRATCH)

print("scratch still there:", SCRATCH.exists())

scratch still there: False


## Recap

- Read a file with `read_csv` and check every column's type with `dtypes` before using it
- Read dates with `parse_dates`, make a column the index with `index_col`, and read part of a file with
  `usecols` and `nrows`
- Keep ids as text with `dtype={"id": str}`, so they keep their leading zeros
- Read other files' conventions with `sep`, `decimal`, `na_values` and `encoding`
- Recognize a byte order mark read with the wrong encoding from a first column named `ï»¿...`

## What is next

The **Inspecting** notebook reads a table once it is loaded, with `head`, `info` and `describe`, and what
each one leaves out.

---

&#8592; **Previous:** [DataFrame](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pandas-deep-dive/03-dataframe.ipynb)  &nbsp;·&nbsp;  [Pandas, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/pandas-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [Inspecting](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pandas-deep-dive/05-inspecting.ipynb) &#8594;
